In [0]:
%python
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
df = spark.table("covid_19_states").toPandas()
print(df.head())




In [0]:
%python
print(df.describe())

In [0]:
%python
print(df.shape)
print(df.isnull().sum())
# print(df.dtypes)
# print(df[df['Tested'].isnull()])

In [0]:
%python
for i in df["Tested"].isna():
  if i == True:
     continue
  else:
    df["positivity_rate"] = df["Confirmed"] / df["Tested"] * 100
    print(df["positivity_rate"].tail(10))
    avg_ratio = df["positivity_rate"].mean()
    print(avg_ratio)
    # df.loc[df["Tested"].isnull(), "Tested"] = df["Confirmed"] / avg_ratio
    # df.drop('positivity_rate', axis=1, inplace=True)    

In [0]:
%python
print(df.isnull().sum())
df["Date"] = pd.to_datetime(df["Date"])
df["month"] = df["Date"].dt.month
df["year"] = df["Date"].dt.year
df["day"] = df["Date"].dt.day
print(df.dtypes)

In [0]:
%python
# # Highest confirmed cases for each state and month (excluding India)
# df_filtered = df[df["State"] != "India"].copy()
# df["Date"] = pd.to_datetime(df["Date"])
# df["month"] = df["Date"].dt.month
# idx = df_filtered.groupby(["State", "month"])["Confirmed"].idxmax()
# monthly_max = df_filtered.loc[idx, ["State", "month", "Confirmed", "Date"]]
# monthly_max = monthly_max.sort_values(["State", "month"])
# display(monthly_max.head(30))
df.drop('Other',axis=1,inplace=True)
print(df.isna().sum())

In [0]:
%python
spark.createDataFrame(df).write.mode("overwrite").saveAsTable("covid_19_states_cleaned")

In [0]:
CREATE OR REPLACE TABLE covid_19_states_cleaned AS
SELECT 
  Date, State, Confirmed, Recovered, Deceased, Tested,
  (Confirmed / Tested) * 100 AS positivity_rate
FROM covid_19_states;

CREATE OR REPLACE TABLE covid_19_states_cleaned AS
SELECT
    Date,
    State,

    Confirmed,
    COALESCE(
        Confirmed - LAG(Confirmed) OVER (PARTITION BY State ORDER BY Date),
        Confirmed
    ) AS daily_confirmed,

    Recovered,
    COALESCE(
        Recovered - LAG(Recovered) OVER (PARTITION BY State ORDER BY Date),
        Recovered
    ) AS daily_recovered,

    Deceased,
    COALESCE(
        Deceased - LAG(Deceased) OVER (PARTITION BY State ORDER BY Date),
        Deceased
    ) AS daily_deceased,

    Tested,
  (Confirmed / Tested) * 100 AS positivity_rate
FROM covid_19_states;

In [0]:
select * from covid_19_states_cleaned where State in ('Maharasthra','India');
-- India Confirmed is not valid or not authentic, like it doesnt matter for it


Total Confirmed, Recoveries and Deceased.

In [0]:
select sum(daily_confirmed) as total_confirmed, sum(daily_recovered) as total_recovered, sum(daily_deceased) as total_deceased from covid_19_states_cleaned where State!='India';

In [0]:
SELECT State, MAX(daily_confirmed) AS max_confirmed
FROM covid_19_states_cleaned
GROUP BY State having State!='India'
ORDER BY max_confirmed DESC; 


Which state has the highest deaths

In [0]:
SELECT State, MAX(daily_deceased) AS max_deceased
FROM covid_19_states_cleaned
GROUP BY State having State!='India'
ORDER BY max_deceased DESC;

Total Cases in India

In [0]:
select State,sum(Confirmed) from covid_19_states_cleaned GROUP BY State having State='India';

Total Recoveries by each state


In [0]:
select State,sum(daily_recovered) as total_rec from covid_19_states_cleaned GROUP BY State HAVING State!='India'
ORDER BY total_rec desc;

In [0]:
select sum(Tested) from covid_19_states_cleaned where State='India';

Months having Biggest Case Spikes.

In [0]:
SELECT month(Date)AS covid_month,State ,MAX(daily_confirmed) AS max_confirmed 
FROM covid_19_states_cleaned 
GROUP BY covid_month,State 
ORDER BY covid_month,State;
-- // Also try to include state to it

States and their Recovery rates

In [0]:
SELECT State, (SUM(daily_recovered)/SUM(daily_confirmed))*100 AS recovery_rate FROM covid_19_states_cleaned GROUP BY State HAVING State!='India' ORDER BY recovery_rate DESC 

States and their death rate

In [0]:
SELECT State, (SUM(daily_deceased)/SUM(daily_confirmed)) * 100 AS death_rate FROM covid_19_states_cleaned GROUP BY State HAVING State!='India' ORDER BY death_rate DESC; 

Death Rate each month for each state

In [0]:
SELECT 
    State,
    MONTH(Date) AS month,
    (SUM(daily_deceased) * 1.0 / SUM(daily_confirmed)) * 100 AS death_rate
FROM covid_19_states_cleaned
WHERE State != 'India'
GROUP BY State, MONTH(Date)
ORDER BY State, month;

Recovery Rate and Death rate each month for each state

In [0]:
SELECT 
    State,
    YEAR(Date) AS year,
    MONTH(Date) AS month,

    -- Monthly increase
    MAX(Confirmed) - MIN(Confirmed) AS monthly_confirmed,
    MAX(Recovered) - MIN(Recovered) AS monthly_recovered,
    MAX(Deceased) - MIN(Deceased) AS monthly_deceased,

    -- Recovery Rate
    ((MAX(Recovered) - MIN(Recovered)) * 1.0 /
     NULLIF(MAX(Confirmed) - MIN(Confirmed), 0)) * 100 AS recovery_rate,

    -- Death Rate
    ((MAX(Deceased) - MIN(Deceased)) * 1.0 /
     NULLIF(MAX(Confirmed) - MIN(Confirmed), 0)) * 100 AS death_rate

FROM covid_19_states_cleaned
WHERE State != 'India'
GROUP BY State, YEAR(Date), MONTH(Date)
ORDER BY State, year, month;

Maximum number of confirmed cases by Month

In [0]:
SELECT covid_month, State, daily_confirmed
FROM (
    SELECT 
        EXTRACT(MONTH FROM Date) AS covid_month,
        State,
        daily_confirmed,
        RANK() OVER (
            PARTITION BY EXTRACT(MONTH FROM Date) 
            ORDER BY daily_confirmed DESC
        ) AS rnk
    FROM covid_19_states_cleaned
    WHERE State != 'India'
) ranked
WHERE rnk = 1
ORDER BY covid_month;

In [0]:
SELECT covid_month, State, Recovered
FROM (
    SELECT 
        EXTRACT(MONTH FROM Date) AS covid_month,
        State,
        Recovered,
        RANK() OVER (
            PARTITION BY EXTRACT(MONTH FROM Date) 
            ORDER BY Recovered DESC
        ) AS rnk
    FROM covid_19_states_cleaned
    WHERE State != 'India'
) ranked
WHERE rnk = 1
ORDER BY covid_month;

In [0]:
select * from covid_19_states_cleaned

In [0]:
SELECT * FROM covid_19_states_cleaned;